A5. Переотчёт паритета бюджета (Singhi 2025, Chen 2024).....


In [1]:
import sys, os, re

def _build_data_tools():
    import json
    import os
    import re
    import warnings
    import numpy as np
    import pandas as pd
    from scipy import stats
    warnings.filterwarnings('ignore')

    def path_join(*parts):
        return os.path.normpath(os.path.join(*parts))

    def normalize_newlines(frame):
        for column in frame.select_dtypes(include=['object', 'string']).columns:
            frame[column] = frame[column].map(lambda value: value.replace('\r\n', '\n').replace('\r', '\n') if isinstance(value, str) else value)
        return frame
    from pathlib import Path
    _start = Path.cwd().resolve()
    _repo = next((p for p in (_start, *_start.parents) if (p / 'experiments/clinical_protocols').is_dir() and (p / 'data').is_dir()), None)
    if _repo is None:
        raise FileNotFoundError('Run the notebook from inside the repository.')
    if os.name == 'nt' and (not str(_repo).startswith(chr(92) * 2 + '?' + chr(92))):
        _repo = Path(chr(92) * 2 + '?' + chr(92) + str(_repo))
    ROOT = str(_repo)
    HERE = str(_repo / 'analyses/budget_efficiency')
    OUT = path_join(HERE, 'results')
    os.makedirs(OUT, exist_ok=True)
    PBASE_DIR = 'experiments/clinical_protocols/delphi_panels/heterogeneous_gpt120_organizer_complete'
    HOMOG_DS_DIR = 'experiments/homogeneous_panels/three_deepseekv4_members_complete'
    NB = 'experiments/clinical_protocols/single_model_baselines/baseline_%s/llm_as_judge/baseline_%s_%s_judge.csv'
    USC = 'experiments/token_budget_controls/universal_self_consistency_gpt_seven_samples/llm_as_judge/usc_gpt_sc%d_judge.csv'
    HOMOG_DIRS = {'gpt': 'experiments/homogeneous_panels/three_gpt_members', 'qwen': 'experiments/homogeneous_panels/three_qwen_members', 'gemini': 'experiments/homogeneous_panels/three_gemini_members'}
    PANELS_OTHER_DIR = {'P-weak': 'experiments/clinical_protocols/delphi_panels/panel_qwen_organizer_gemini_deepseek_pro', 'P-mixed': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_gemini_glm', 'P-strong': 'experiments/clinical_protocols/delphi_panels/panel_deepseek_organizer_glm_gpt120'}
    PANEL_FILE = {'P-weak': 'delphi_pweak_noroles_judge.csv', 'P-mixed': 'delphi_pmixed_noroles_judge.csv', 'P-strong': 'delphi_pstrong_noroles_judge.csv'}
    SINGLES = ['gpt', 'qwen', 'deepseekv4', 'deepseekv4pro', 'gemini', 'glm', 'kimi']
    COMPOSITION = {'P-base': ['gpt', 'qwen', 'deepseekv4'], 'P-weak': ['qwen', 'gemini', 'deepseekv4pro'], 'P-mixed': ['deepseekv4', 'gemini', 'glm'], 'P-strong': ['deepseekv4', 'glm', 'gpt'], 'homog gpt': ['gpt'] * 3, 'homog qwen': ['qwen'] * 3, 'homog deepseekv4': ['deepseekv4'] * 3, 'homog gemini': ['gemini'] * 3}
    HETERO = {'P-base': 1, 'P-weak': 1, 'P-mixed': 1, 'P-strong': 1, 'homog gpt': 0, 'homog qwen': 0, 'homog deepseekv4': 0, 'homog gemini': 0}
    PANELS = list(COMPOSITION)
    LADDER = ['baseline gpt', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'homog gpt', 'homog deepseekv4', 'P-base']

    def arm_paths(design='noroles'):
        p = {}
        for m in SINGLES:
            p['baseline ' + m] = NB % (m, design, m)
        if design == 'noroles':
            for k in (3, 5, 7):
                p['USC@%d gpt' % k] = USC % k
            for tag, d in HOMOG_DIRS.items():
                p['homog ' + tag] = d + '/llm_as_judge/delphi_homog_%s_noroles_judge.csv' % tag
            p['homog deepseekv4'] = HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'
            for k, d in PANELS_OTHER_DIR.items():
                p[k] = d + '/llm_as_judge/' + PANEL_FILE[k]
        p['P-base'] = PBASE_DIR + '/llm_as_judge/delphi_base_%s_judge.csv' % design
        return p
    ROUNDS_DIR = {'P-base': (PBASE_DIR, 'delphi_base_noroles_protocol_r%d_judge.csv'), 'P-weak': (PANELS_OTHER_DIR['P-weak'], 'delphi_pweak_noroles_protocol_r%d_judge.csv'), 'P-mixed': (PANELS_OTHER_DIR['P-mixed'], 'delphi_pmixed_noroles_protocol_r%d_judge.csv'), 'P-strong': (PANELS_OTHER_DIR['P-strong'], 'delphi_pstrong_noroles_protocol_r%d_judge.csv'), 'homog gpt': (HOMOG_DIRS['gpt'], 'delphi_homog_gpt_noroles_protocol_r%d_judge.csv'), 'homog gemini': (HOMOG_DIRS['gemini'], 'delphi_homog_gemini_noroles_protocol_r%d_judge.csv'), 'homog deepseekv4': (HOMOG_DS_DIR, 'delphi_homog_deepseekv4_noroles_protocol_r%d_judge.csv')}
    RETEST = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_retest_r%d_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_retest_r%d_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_retest_r%d_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_retest_r%d_judge.csv'}
    SHORT = {'P-base': PBASE_DIR + '/llm_as_judge/judge_robustness/delphi_base_%s_short_judge.csv', 'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_robustness/baseline_%s_gpt_short_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_robustness/baseline_%s_qwen_short_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_robustness/baseline_%s_deepseekv4_short_judge.csv'}
    JUDGE41 = {'baseline gpt': 'experiments/clinical_protocols/single_model_baselines/baseline_gpt/llm_as_judge/judge_gpt41/baseline_%s_gpt_judge.csv', 'baseline qwen': 'experiments/clinical_protocols/single_model_baselines/baseline_qwen/llm_as_judge/judge_gpt41/baseline_%s_qwen_judge.csv', 'baseline deepseekv4': 'experiments/clinical_protocols/single_model_baselines/baseline_deepseekv4/llm_as_judge/judge_gpt41/baseline_%s_deepseekv4_judge.csv'}
    CASE_COVARS = ['Type', 'Sub Clinical Specialties', 'Age Group', 'Genealogy', 'Diagnosis10', 'Vignette_race']
    DEXP = 'experiments/panel_parameter_sweeps'
    SWEEPS = {'experts': (DEXP + '/experts/llm_as_judge', 'delphi_experts_%s_judge.csv'), 'questionnaire_items': (DEXP + '/questionnaire_items/llm_as_judge', 'delphi_questionnaire_items_%s_judge.csv'), 'questions_per_expert': (DEXP + '/questions_per_expert/llm_as_judge', 'delphi_questions_per_expert_%s_judge.csv'), 'temperature': (DEXP + '/temperature/llm_as_judge', 'delphi_temperature_%s_judge.csv'), 'rounds': (DEXP + '/rounds/llm_as_judge', 'delphi_%s_judge.csv')}
    EXPERT_GROUPS = {'group1_10models': 10, 'group2_4models': 4, 'group3_5models': 5, 'group4_5models': 5, 'group5_7models': 7, 'group6_5models': 5, 'group7_6models': 6}
    EXT = 'experiments'
    MEDMCQA = {'baseline deepseekv4': EXT + '/external_benchmarks/medmcqa/baseline_deepseekv4/baseline_medmcqa_deepseekv4.csv', 'baseline gpt': EXT + '/external_benchmarks/medmcqa/baseline_gpt/baseline_medmcqa_gpt.csv', 'baseline qwen': EXT + '/external_benchmarks/medmcqa/baseline_qwen/baseline_medmcqa_qwen.csv', 'baseline alice': EXT + '/external_benchmarks/initial_runs/baseline_alice/baseline_medmcqa_alice.csv', 'baseline qwen36': EXT + '/external_benchmarks/initial_runs/baseline_qwen36/baseline_medmcqa_qwen36.csv', 'P-ac (gpt/qwen/ds)': EXT + '/external_benchmarks/medmcqa/panel_gpt120_organizer_qwen_deepseek/delphi_medmcqa_noroles.csv', 'P-ac (gpt/qwen/alice)': EXT + '/external_benchmarks/initial_runs/panel_gpt120_organizer_qwen_alice/delphi_medmcqa_noroles.csv', 'P-deepseektrio': EXT + '/external_benchmarks/initial_runs/three_deepseek_members/delphi_medmcqa_noroles.csv', 'P-base-like': EXT + '/external_benchmarks/initial_runs/heterogeneous_gpt120_organizer/delphi_medmcqa_noroles.csv'}

    def rd_any(rel, index=None):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        with open(p, encoding='utf-8-sig', errors='replace') as fh:
            head = fh.readline()
        sep = ';' if head.count(';') > head.count(',') else ','
        d = normalize_newlines(pd.read_csv(p, sep=sep, encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if index and index in d.columns:
            d = d.dropna(subset=[index]).drop_duplicates(index).set_index(index)
        return d

    def sweep_arms(kind, verbose=True):
        import glob as _g
        folder, _ = SWEEPS[kind]
        out = {}
        for p in sorted(_g.glob(path_join(ROOT, folder, '*_judge.csv'))):
            name = os.path.basename(p)[:-len('_judge.csv')]
            d = rd(os.path.relpath(p, ROOT).replace('\\', '/'))
            if d is not None:
                out[name] = d
        if verbose:
            print('свип %s: плеч %d, кейсов %d' % (kind, len(out), len(next(iter(out.values()))) if out else 0))
        return out

    def member_triage_acc(d):
        t = true_label(d)
        out = {}
        for p in participants(d):
            lab = d['r1_emergency__' + p].map(norm_label)
            ok = lab.notna() & t.notna()
            out[p] = float((lab[ok] == t[ok]).mean()) if ok.any() else np.nan
        return pd.Series(out)

    def rd(rel, need_score=True):
        p = path_join(ROOT, rel)
        if not os.path.exists(p):
            return None
        d = normalize_newlines(pd.read_csv(p, sep=';', encoding='utf-8-sig', low_memory=False))
        if 'table_source' in d.columns:
            d = d[d['table_source'].fillna('result_df') == 'result_df']
        if need_score and 'Judge_score' not in d.columns:
            return None
        sub = ['Judge_score', 'Vignette'] if need_score else ['Vignette']
        d = d.dropna(subset=sub).drop_duplicates('Vignette', keep='first')
        return d.set_index('Vignette')

    def load_arms(design='noroles', verbose=True):
        a = {}
        for name, rel in arm_paths(design).items():
            d = rd(rel)
            if d is not None:
                a[name] = d
        if verbose:
            print('плеч загружено: %d' % len(a))
            print('   ' + ', '.join(a))
        return a

    def js(arms, name):
        return pd.to_numeric(arms[name]['Judge_score'], errors='coerce').dropna()

    def score_matrix(arms, names=None):
        names = list(arms) if names is None else names
        m = pd.concat([js(arms, n).rename(n) for n in names], axis=1)
        return m.dropna()

    def meta(arms=None):
        d = rd(PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv')
        cols = [c for c in CASE_COVARS if c in d.columns]
        return d[cols].copy()

    def item_frame(d):
        r = {}
        if d is None or 'judge_details' not in d.columns:
            return r
        for v, s in d['judge_details'].dropna().items():
            try:
                arr = json.loads(s)
            except Exception:
                continue
            for it in arr:
                ax = next((t.split(':', 1)[1] for t in it.get('tags') or [] if isinstance(t, str) and t.startswith('axis:')), None)
                r[v, it['id']] = {'met': bool(it.get('met')), 'points': it.get('points', 0) or 0, 'axis': ax}
        return r

    def load_items(design='noroles', arms_wanted=None, verbose=True):
        paths = arm_paths(design)
        if arms_wanted:
            paths = {k: v for k, v in paths.items() if k in arms_wanted}
        items = {}
        for name, rel in paths.items():
            d = rd(rel)
            f = item_frame(d)
            if f:
                items[name] = f
        common = None
        for f in items.values():
            common = set(f) if common is None else common & set(f)
        ref = next(iter(items.values()))
        pos = sorted([k for k in common or set() if ref[k]['points'] > 0])
        if verbose:
            print('плеч с попунктными вердиктами: %d' % len(items))
            print('общих положительных пунктов: %d' % len(pos))
        return (items, pos)

    def items_long(items, pos, arms_wanted=None):
        names = arms_wanted or list(items)
        rows = []
        for a in names:
            f = items[a]
            for k in pos:
                it = f[k]
                rows.append({'case': k[0], 'cid': k[1], 'arm': a, 'met': int(it['met']), 'points': it['points'], 'axis': it['axis']})
        return pd.DataFrame(rows)

    def paired(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(float), b.loc[ix].astype(float))
        d = x - y
        p = stats.wilcoxon(x, y)[1] if (d != 0).any() else np.nan
        t = stats.ttest_rel(x, y)
        return {'n': len(ix), 'delta': float(d.mean()), 'sd': float(d.std(ddof=1)), 'share_gt': float((d > 0).mean()), 'p_wilcoxon': float(p), 't': float(t.statistic), 'p_ttest': float(t.pvalue)}

    def se_clt(x):
        x = np.asarray(x, float)
        n = len(x)
        return float(np.sqrt(x.var(ddof=1) / n))

    def se_clustered(x, cluster, cr1=True):
        x = np.asarray(x, float)
        cl = np.asarray(cluster)
        n = len(x)
        e = x - x.mean()
        df = pd.DataFrame({'e': e, 'c': cl})
        r = df.groupby('c', observed=True)['e'].sum().to_numpy()
        ss = df.groupby('c', observed=True)['e'].apply(lambda s: float((s ** 2).sum())).to_numpy()
        cross = float((r ** 2 - ss).sum())
        v = se_clt(x) ** 2 + cross / n ** 2
        G = len(r)
        if cr1 and G > 1:
            v *= G / (G - 1.0)
        return (float(np.sqrt(max(v, 0.0))), G)

    def cluster_test(d, cluster, cr1=True):
        d = pd.Series(d).astype(float)
        cl = pd.Series(cluster).reindex(d.index)
        ok = cl.notna()
        d, cl = (d[ok], cl[ok])
        m = float(d.mean())
        s0 = se_clt(d)
        s1, G = se_clustered(d, cl, cr1=cr1)
        dfree = max(G - 1, 1)
        t = m / s1 if s1 > 0 else np.nan
        p = float(2 * stats.t.sf(abs(t), dfree)) if np.isfinite(t) else np.nan
        tc = stats.t.ppf(0.975, dfree)
        return {'n': len(d), 'G': G, 'mean': m, 'se_naive': s0, 'se_cluster': s1, 'DEFF': (s1 / s0) ** 2 if s0 > 0 else np.nan, 't': float(t), 'df': dfree, 'p': p, 'ci_lo': m - tc * s1, 'ci_hi': m + tc * s1, 'p_naive': float(2 * stats.norm.sf(abs(m / s0))) if s0 > 0 else np.nan}

    def icc_anova(values, groups):
        df = pd.DataFrame({'y': np.asarray(values, float), 'g': np.asarray(groups)})
        df = df.dropna()
        grp = df.groupby('g', observed=True)['y']
        ni = grp.size().to_numpy().astype(float)
        k = len(ni)
        N = ni.sum()
        if k < 2 or N <= k:
            return (0.0, float(ni.mean() if k else 1), k)
        gm = df['y'].mean()
        msb = float((ni * (grp.mean().to_numpy() - gm) ** 2).sum() / (k - 1))
        ssw = float(((df['y'] - df['g'].map(grp.mean()).to_numpy()) ** 2).sum())
        msw = ssw / (N - k)
        B0 = float((N - (ni ** 2).sum() / N) / (k - 1))
        rho = (msb - msw) / (msb + (B0 - 1) * msw) if msb + (B0 - 1) * msw > 0 else 0.0
        return (float(np.clip(rho, 0, 1)), B0, k)

    def ci_wald(p, n, z=1.96):
        se = np.sqrt(p * (1 - p) / n)
        return (p - z * se, p + z * se)

    def ci_wilson(p, n, z=1.96):
        d = 1 + z ** 2 / n
        c = (p + z ** 2 / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
        return (c - h, c + h)

    def ci_agresti_coull(x, n, z=1.96):
        nt = n + z ** 2
        pt = (x + z ** 2 / 2) / nt
        se = np.sqrt(pt * (1 - pt) / nt)
        return (pt - z * se, pt + z * se)

    def ci_clopper_pearson(x, n, alpha=0.05):
        lo = 0.0 if x == 0 else stats.beta.ppf(alpha / 2, x, n - x + 1)
        hi = 1.0 if x == n else stats.beta.ppf(1 - alpha / 2, x + 1, n - x)
        return (float(lo), float(hi))

    def boot_ci(x, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        x = np.asarray(x, float)
        n = len(x)
        s = np.array([stat(x[rng.integers(0, n, n)]) for _ in range(B)])
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def boot_ci_cluster(x, cluster, stat=np.mean, B=5000, seed=0, alpha=0.05):
        rng = np.random.default_rng(seed)
        df = pd.DataFrame({'x': np.asarray(x, float), 'c': np.asarray(cluster)})
        groups = [g['x'].to_numpy() for _, g in df.groupby('c', observed=True)]
        G = len(groups)
        s = []
        for _ in range(B):
            pick = rng.integers(0, G, G)
            s.append(stat(np.concatenate([groups[i] for i in pick])))
        s = np.array(s)
        return (float(np.quantile(s, alpha / 2)), float(np.quantile(s, 1 - alpha / 2)))

    def holm(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 0.0
        for i, k in enumerate(order):
            run = max(run, (m - i) * p[k])
            adj[k] = min(run, 1.0)
        return adj

    def bh(pvals):
        p = np.asarray(pvals, float)
        m = len(p)
        order = np.argsort(p)
        adj = np.empty(m)
        run = 1.0
        for i in range(m - 1, -1, -1):
            k = order[i]
            run = min(run, m * p[k] / (i + 1))
            adj[k] = min(run, 1.0)
        return adj

    def bonferroni(pvals):
        p = np.asarray(pvals, float)
        return np.minimum(p * len(p), 1.0)

    def mcnemar(a, b):
        ix = a.index.intersection(b.index)
        x, y = (a.loc[ix].astype(bool), b.loc[ix].astype(bool))
        n01, n10 = (int((~x & y).sum()), int((x & ~y).sum()))
        if n01 + n10 == 0:
            return {'n01': n01, 'n10': n10, 'p': 1.0}
        return {'n01': n01, 'n10': n10, 'p': float(stats.binomtest(n10, n01 + n10, 0.5).pvalue)}
    ORD = {'SELF': 0, 'NONEMERG': 1, 'EMERG': 2}
    GT = {'Self-care appropriate': 'SELF', 'Requires non-emergent care': 'NONEMERG', 'Requires emergent care': 'EMERG'}
    _DASH = dict.fromkeys(map(ord, '‐‑‒–—―−\xad'), '-')
    _KEYS = [('requires emergent care', 'EMERG'), ('requires non-emergent care', 'NONEMERG'), ('self-care appropriate', 'SELF')]

    def norm_label(x):
        if not isinstance(x, str) or not x.strip():
            return None
        if x.strip() in GT:
            return GT[x.strip()]
        import unicodedata
        s = unicodedata.normalize('NFKC', x).translate(_DASH).lower()
        s = re.sub('\\s+', ' ', re.sub('[^a-z\\- ]', ' ', s)).strip()
        for k, v in _KEYS:
            if s.startswith(k) or k in s:
                return v
        if 'non-emergent' in s or 'non emergent' in s:
            return 'NONEMERG'
        if 'emergent' in s or 'emergency' in s:
            return 'EMERG'
        if 'self' in s:
            return 'SELF'
        return None
    _PAT = re.compile('emergency\\s*type\\s*[:\\-]\\s*[\\*_`#\\s]*([^\\n\\r\\*_|]{0,90})', re.I)

    def parse_triage(text):
        if not isinstance(text, str) or not text.strip():
            return None
        import unicodedata
        m = _PAT.search(unicodedata.normalize('NFKC', text).translate(_DASH))
        if not m:
            return None
        return norm_label('Requires ' + m.group(1) if False else m.group(1))

    def pred_triage(d):
        if 'delphi_consensus_emergency_type' in d.columns:
            s = d['delphi_consensus_emergency_type'].map(norm_label)
            if s.notna().mean() > 0.5:
                return s
        for c in ('winner_protocol', 'baseline_protocol_anonymous'):
            if c in d.columns:
                s = d[c].map(parse_triage)
                if s.notna().mean() > 0.3:
                    return s
        return pd.Series(index=d.index, dtype=object)

    def participants(d):
        return sorted((c.split('__', 1)[1] for c in d.columns if c.startswith('r1_emergency__')))

    def true_label(d):
        return d['Type'].map(GT)

    def words(s):
        return len(re.findall('\\S+', str(s))) if isinstance(s, str) else np.nan

    def fmt_p(p):
        if p is None or (isinstance(p, float) and (not np.isfinite(p))):
            return '  n/a '
        return '%.3f' % p if p >= 0.001 else '%.1e' % p

    def show(df, title=None, floatfmt='%.4f'):
        if title:
            print('\n' + title)
            print('-' * max(len(title), 60))
        with pd.option_context('display.width', 200, 'display.max_columns', 60, 'display.float_format', lambda v: floatfmt % v):
            print(df)

    def save(df, name):
        p = path_join(OUT, name)
        df.to_csv(p, sep=';', encoding='utf-8-sig')
        print('сохранено: out/%s  (%d строк)' % (name, len(df)))
        return p
    return locals()
globals().update({k: v for k, v in _build_data_tools().items() if not k.startswith('_')})
del _build_data_tools
A = load_arms()


плеч загружено: 18
   baseline gpt, baseline qwen, baseline deepseekv4, baseline deepseekv4pro, baseline gemini, baseline glm, baseline kimi, USC@3 gpt, USC@5 gpt, USC@7 gpt, homog gpt, homog qwen, homog gemini, homog deepseekv4, P-weak, P-mixed, P-strong, P-base


In [2]:
def panel_budget(rel):
    d = rd(rel)
    gen_t = [c for c in d.columns if re.match('r[123]_total_tokens__', c)]
    gen_c = [c for c in d.columns if re.match('r[123]_completion_tokens__', c)]
    gen_p = [c for c in d.columns if re.match('r[123]_prompt_tokens__', c)]
    agg_t = [c for c in ['organizer_after_r1_total_tokens', 'organizer_after_r2_total_tokens', 'final_synthesis_total_tokens'] if c in d.columns]
    agg_c = [c.replace('total', 'completion') for c in agg_t if c.replace('total', 'completion') in d.columns]
    agg_p = [c.replace('total', 'prompt') for c in agg_t if c.replace('total', 'prompt') in d.columns]
    f = lambda cols: d[cols].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    return pd.DataFrame({'S_total': f(gen_t), 'S_compl': f(gen_c), 'S_prompt': f(gen_p), 'V_total': f(agg_t), 'V_compl': f(agg_c), 'V_prompt': f(agg_p)}, index=d.index)
PANEL_REL = {'P-base': PBASE_DIR + '/llm_as_judge/delphi_base_noroles_judge.csv', 'P-weak': PANELS_OTHER_DIR['P-weak'] + '/llm_as_judge/' + PANEL_FILE['P-weak'], 'P-mixed': PANELS_OTHER_DIR['P-mixed'] + '/llm_as_judge/' + PANEL_FILE['P-mixed'], 'P-strong': PANELS_OTHER_DIR['P-strong'] + '/llm_as_judge/' + PANEL_FILE['P-strong'], 'homog gpt': HOMOG_DIRS['gpt'] + '/llm_as_judge/delphi_homog_gpt_noroles_judge.csv', 'homog qwen': HOMOG_DIRS['qwen'] + '/llm_as_judge/delphi_homog_qwen_noroles_judge.csv', 'homog gemini': HOMOG_DIRS['gemini'] + '/llm_as_judge/delphi_homog_gemini_noroles_judge.csv', 'homog deepseekv4': HOMOG_DS_DIR + '/llm_as_judge/delphi_homog_deepseekv4_noroles_judge.csv'}
BUD = {}
for k, rel in PANEL_REL.items():
    try:
        BUD[k] = panel_budget(rel)
    except Exception as e:
        print('пропуск', k, e)


In [3]:
def usc_budget(k):
    d = rd(USC % k)
    g = lambda c: pd.to_numeric(d[c], errors='coerce').fillna(0) if c in d.columns else 0
    return pd.DataFrame({'S_total': g('draft_total_tokens'), 'S_compl': np.nan, 'S_prompt': np.nan, 'V_total': g('usc_total_tokens'), 'V_compl': g('usc_completion_tokens'), 'V_prompt': g('usc_prompt_tokens')}, index=d.index)

def base_budget(m):
    d = rd(NB % (m, 'noroles', m))
    g = lambda c: pd.to_numeric(d[c], errors='coerce').fillna(0)
    return pd.DataFrame({'S_total': g('total_tokens'), 'S_compl': g('completion_tokens'), 'S_prompt': g('prompt_tokens'), 'V_total': 0.0, 'V_compl': 0.0, 'V_prompt': 0.0}, index=d.index)
for k in (3, 5, 7):
    BUD['USC@%d gpt' % k] = usc_budget(k)
for m in ['gpt', 'qwen', 'deepseekv4', 'gemini', 'glm', 'deepseekv4pro']:
    BUD['baseline ' + m] = base_budget(m)
rows = []
for name, b in BUD.items():
    if name not in A:
        continue
    q = float(js(A, name).mean())
    rows.append({'плечо': name, 'Judge': q, 'S (генерация)': float(b['S_total'].mean()), 'V (агрегация)': float(b['V_total'].mean()), 'total': float((b['S_total'] + b['V_total']).mean()), 'доля V, %': 100 * float(b['V_total'].mean()) / max(float((b['S_total'] + b['V_total']).mean()), 1), 'completion': float((b['S_compl'].fillna(0) + b['V_compl']).mean())})
B = pd.DataFrame(rows).set_index('плечо').sort_values('total')
show(B, 'Бюджет на кейс: генерация S против агрегации V')



Бюджет на кейс: генерация S против агрегации V
------------------------------------------------------------
                        Judge  S (генерация)  V (агрегация)      total  доля V, %  completion
плечо                                                                                        
homog qwen             0.6979       111.2009       226.6114   337.8122    67.0820    118.9520
baseline qwen          0.5622      1250.7904         0.0000  1250.7904     0.0000    452.3319
baseline glm           0.5870      1377.6288         0.0000  1377.6288     0.0000    787.1528
baseline gpt           0.6258      1752.0087         0.0000  1752.0087     0.0000    914.5022
baseline deepseekv4    0.6401      1811.7719         0.0000  1811.7719     0.0000   1044.6667
baseline deepseekv4pro 0.5601      1850.8559         0.0000  1850.8559     0.0000   1173.3886
baseline gemini        0.4247      1935.4760         0.0000  1935.4760     0.0000   1325.0786
homog gemini           0.7172      6391.7467 

In [4]:
LAM = [0, 0.5, 1, 2, 3, 5]
rows = []
for name, b in BUD.items():
    if name not in A:
        continue
    r = {'плечо': name, 'Judge': float(js(A, name).mean())}
    for lam in LAM:
        r['C(λ=%g)' % lam] = float((b['S_total'] + lam * b['V_total']).mean())
    rows.append(r)
C = pd.DataFrame(rows).set_index('плечо')
show(C, 'Стоимость на кейс при разной цене верификации λ')



Стоимость на кейс при разной цене верификации λ
------------------------------------------------------------
                        Judge     C(λ=0)   C(λ=0.5)     C(λ=1)     C(λ=2)     C(λ=3)      C(λ=5)
плечо                                                                                           
P-base                 0.7899 14537.0131 20641.7336 26746.4541 38955.8952 51165.3362  75584.2183
P-weak                 0.7453 14925.0742 19662.7009 24400.3275 33875.5808 43350.8341  62301.3406
P-mixed                0.7864 11331.4236 17044.7271 22758.0306 34184.6376 45611.2445  68464.4585
P-strong               0.8061 11624.0917 17658.4410 23692.7904 35761.4891 47830.1878  71967.5852
homog gpt              0.7588 10829.3930 15894.5742 20959.7555 31090.1179 41220.4803  61481.2052
homog qwen             0.6979   111.2009   224.5066   337.8122   564.4236   791.0349   1244.2576
homog gemini           0.7172  6391.7467  9536.5852 12681.4236 18971.1004 25260.7773  37840.1310
homog deepseekv4 

In [5]:
rows = []
for lam in LAM:
    p = float((BUD['P-base']['S_total'] + lam * BUD['P-base']['V_total']).mean())
    u = float((BUD['USC@7 gpt']['S_total'] + lam * BUD['USC@7 gpt']['V_total']).mean())
    rows.append({'λ': lam, 'P-base C': p, 'USC@7 C': u, 'запас USC над панелью, %': 100 * (u - p) / p, 'панель дешевле': u > p})
show(pd.DataFrame(rows).set_index('λ'), 'Паритет бюджета в зависимости от λ')



Паритет бюджета в зависимости от λ
------------------------------------------------------------
         P-base C     USC@7 C  запас USC над панелью, %  панель дешевле
λ                                                                      
0.0000 14537.0131   6228.9869                  -57.1508           False
0.5000 20641.7336  16875.2948                  -18.2467           False
1.0000 26746.4541  27521.6026                    2.8981            True
2.0000 38955.8952  48814.2183                   25.3064            True
3.0000 51165.3362  70106.8341                   37.0202            True
5.0000 75584.2183 112692.0655                   49.0947            True


In [6]:
rows = []
for name in ['P-base', 'P-weak', 'P-mixed', 'P-strong', 'homog gpt', 'homog deepseekv4', 'USC@3 gpt', 'USC@5 gpt', 'USC@7 gpt', 'baseline gpt']:
    b = BUD.get(name)
    if b is None or name not in A:
        continue
    tot = float((b['S_total'] + b['V_total']).mean())
    cmp_tok = float((b['S_compl'].fillna(0) + b['V_compl']).mean())
    rows.append({'плечо': name, 'Judge': float(js(A, name).mean()), 'total': tot, 'completion': cmp_tok, 'доля completion, %': 100 * cmp_tok / tot if tot else np.nan})
CT = pd.DataFrame(rows).set_index('плечо')
CT['completion / USC@7'] = CT['completion'] / CT.loc['USC@7 gpt', 'completion']
show(CT, 'Паритет по completion-токенам')
print()
print('ВЫВОД, КОТОРЫЙ НАДО ПРИНЯТЬ ЧЕСТНО:')
print('  по total-токенам   панель дешевле USC@7 на ~3%;')
print('  по completion-токенам панель дороже USC@7 в %.1f раза.' % CT.loc['P-base', 'completion / USC@7'])
print()
print('Это не ошибка счёта, а следствие разной структуры бюджета: у USC@7 доля')
print('completion = %.0f%%, у панели = %.0f%%. Утверждение «панель дешевле»' % (CT.loc['USC@7 gpt', 'доля completion, %'], CT.loc['P-base', 'доля completion, %']))
print('верно РОВНО для метрики total-токенов и неверно для метрики completion.')
print()
print('Что с этим делать в статье (варианты, по убыванию честности):')
print('  1) отчитаться обеими метриками и сместить акцент с «дешевле» на')
print('     «сопоставимо по total и заведомо не выигрывает за счёт большего')
print('     числа сэмплов» — кривая USC плоская, и это главный аргумент;')
print('  2) добавить плечо USC с бюджетом, равным панели ПО COMPLETION')
print('     (k примерно 25-30) — это требует новых прогонов;')
print('  3) обосновать total-метрику ценами конкретного провайдера и привести')
print('     расчёт в деньгах.')
print()
print('ДЕФЕКТ ДАННЫХ: у плеча homog qwen учёт токенов явно сломан')
print('(S = %.0f, V = %.0f на кейс против ~11000/12000 у прочих панелей).' % (BUD['homog qwen']['S_total'].mean(), BUD['homog qwen']['V_total'].mean()))
print('Это плечо нельзя использовать ни в каком бюджетном сравнении, пока')
print('не выяснена причина; на Judge_score дефект не влияет.')



Паритет по completion-токенам
------------------------------------------------------------
                  Judge      total  completion  доля completion, %  completion / USC@7
плечо                                                                                 
P-base           0.7899 26746.4541  11962.0830             44.7240              4.2066
P-weak           0.7453 24400.3275  11335.0961             46.4547              3.9861
P-mixed          0.7864 22758.0306  12631.5459             55.5037              4.4421
P-strong         0.8061 23692.7904  12823.9738             54.1261              4.5097
homog gpt        0.7588 20959.7555   9724.4105             46.3956              3.4197
homog deepseekv4 0.7811 23911.5284  12918.9563             54.0281              4.5431
USC@3 gpt        0.6177 17020.5240   2067.1179             12.1449              0.7269
USC@5 gpt        0.6284 18754.6943   2060.2751             10.9854              0.7245
USC@7 gpt        0.6280 27521.6026   2

In [7]:
rows = []
for name, b in BUD.items():
    if name not in A:
        continue
    s, v = (float(b['S_total'].mean()), float(b['V_total'].mean()))
    rows.append({'плечо': name, 'Scaling budget': s, 'Control budget': v, 'Control/Scaling': v / s if s else np.nan, 'Performance': float(js(A, name).mean()), 'Performance на 10k токенов': float(js(A, name).mean()) / ((s + v) / 10000.0)})
T = pd.DataFrame(rows).set_index('плечо').sort_values('Performance', ascending=False)
show(T, 'Тройка бюджетов Chen 2024')



Тройка бюджетов Chen 2024
------------------------------------------------------------
                        Scaling budget  Control budget  Control/Scaling  Performance  Performance на 10k токенов
плечо                                                                                                           
P-strong                    11624.0917      12068.6987           1.0382       0.8061                      0.3402
P-base                      14537.0131      12209.4410           0.8399       0.7899                      0.2953
P-mixed                     11331.4236      11426.6070           1.0084       0.7864                      0.3455
homog deepseekv4            11455.0349      12456.4934           1.0874       0.7811                      0.3266
homog gpt                   10829.3930      10130.3624           0.9355       0.7588                      0.3620
P-weak                      14925.0742       9475.2533           0.6349       0.7453                      0.3054
homog ge

In [8]:
pts = []
b1 = BUD['baseline gpt']
pts.append({'k': 1, 'tokens': float((b1['S_total'] + b1['V_total']).mean()), 'Judge': float(js(A, 'baseline gpt').mean())})
for k in (3, 5, 7):
    b = BUD['USC@%d gpt' % k]
    pts.append({'k': k, 'tokens': float((b['S_total'] + b['V_total']).mean()), 'Judge': float(js(A, 'USC@%d gpt' % k).mean())})
U = pd.DataFrame(pts)
show(U.set_index('k'), 'Кривая USC: бюджет против качества')
pb = float((BUD['P-base']['S_total'] + BUD['P-base']['V_total']).mean())
slope, intercept = np.polyfit(U['tokens'], U['Judge'], 1)
pred = intercept + slope * pb
print()
print('бюджет панели P-base            : %8.0f токенов/кейс' % pb)
print('бюджет USC@7                    : %8.0f токенов/кейс' % U.loc[U.k == 7, 'tokens'].iloc[0])
print('наклон кривой USC               : %+.3e Judge на токен' % slope)
print('прогноз USC при бюджете панели  : %.4f' % pred)
print('фактический P-base              : %.4f' % float(js(A, 'P-base').mean()))
print('превышение панели над кривой USC: %+.4f' % (float(js(A, 'P-base').mean()) - pred))
print()
print('Кривая USC статистически плоская (см. A2, семейство F5: ни один')
print('контраст USC@k против baseline не значим), поэтому экстраполяция даже')
print('на в разы больший бюджет не меняет вывода: панель выигрывает не бюджетом.')



Кривая USC: бюджет против качества
------------------------------------------------------------
      tokens  Judge
k                  
1  1752.0087 0.6258
3 17020.5240 0.6177
5 18754.6943 0.6284
7 27521.6026 0.6280

бюджет панели P-base            :    26746 токенов/кейс
бюджет USC@7                    :    27522 токенов/кейс
наклон кривой USC               : +7.068e-08 Judge на токен
прогноз USC при бюджете панели  : 0.6257
фактический P-base              : 0.7899
превышение панели над кривой USC: +0.1643

Кривая USC статистически плоская (см. A2, семейство F5: ни один
контраст USC@k против baseline не значим), поэтому экстраполяция даже
на в разы больший бюджет не меняет вывода: панель выигрывает не бюджетом.


In [9]:
rows = []
for w in [1, 2, 3, 4, 5]:
    r = {'w (completion/prompt)': w}
    for name in ['P-base', 'USC@7 gpt', 'homog gpt', 'baseline gpt']:
        b = BUD.get(name)
        if b is None:
            continue
        pr = float((b['S_prompt'].fillna(0) + b['V_prompt']).mean())
        cm = float((b['S_compl'].fillna(0) + b['V_compl']).mean())
        if pr == 0 and cm == 0:
            r[name] = np.nan
        else:
            r[name] = pr + w * cm
    rows.append(r)
W = pd.DataFrame(rows).set_index('w (completion/prompt)')
show(W, 'Взвешенный бюджет (prompt + w·completion)')
print()
print('ПРИМЕЧАНИЕ: у USC S_prompt/S_compl по черновикам в данных не разложены,')
print('поэтому строка USC@7 здесь учитывает только вызов отбора и НЕ сопоставима')
print('напрямую. Для статьи это надо либо дособрать, либо отчитываться по total.')



Взвешенный бюджет (prompt + w·completion)
------------------------------------------------------------
                          P-base  USC@7 gpt  homog gpt  baseline gpt
w (completion/prompt)                                               
1                     26746.4541 21292.6157 20959.7555     1752.0087
2                     38708.5371 24136.2445 30684.1659     2666.5109
3                     50670.6201 26979.8734 40408.5764     3581.0131
4                     62632.7031 29823.5022 50132.9869     4495.5153
5                     74594.7860 32667.1310 59857.3974     5410.0175

ПРИМЕЧАНИЕ: у USC S_prompt/S_compl по черновикам в данных не разложены,
поэтому строка USC@7 здесь учитывает только вызов отбора и НЕ сопоставима
напрямую. Для статьи это надо либо дособрать, либо отчитываться по total.


In [10]:
save(B.reset_index(), 'generation_and_aggregation_budget_parity_budget_decomposition.csv')
save(C.reset_index(), 'generation_and_aggregation_budget_parity_lambda_sensitivity.csv')
save(T.reset_index(), 'generation_and_aggregation_budget_parity_chen_triple.csv')
save(U, 'generation_and_aggregation_budget_parity_usc_curve.csv')


сохранено: out/generation_and_aggregation_budget_parity_budget_decomposition.csv  (17 строк)
сохранено: out/generation_and_aggregation_budget_parity_lambda_sensitivity.csv  (17 строк)
сохранено: out/generation_and_aggregation_budget_parity_chen_triple.csv  (17 строк)
сохранено: out/generation_and_aggregation_budget_parity_usc_curve.csv  (4 строк)


'\\\\?\\D:\\research_llmfmed\\research_llmfmed\\1design_alpha\\dataverse_gold\\github_repository\\analyses\\budget_efficiency\\results\\generation_and_aggregation_budget_parity_usc_curve.csv'